# Lab 2 — NumPy Arrays & Vectorized Math

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Learning objectives

After this notebook you will be able to:

1. Create **1-D NumPy arrays** from Python lists.
2. Apply **vectorized** operations (no explicit `for` loops).
3. **Normalize** numeric features using mean and standard deviation.
4. Build a **2-D matrix** with `column_stack` and compute **axis** statistics.

> **Checkpoints:** `votes.shape == (5,)`, `matrix.shape == (5, 2)`, column means ≈ **[1392, 1290]**.

**Companion script:** `../scripts/lab02_numpy_arrays.py`


## Why NumPy underpins data science

| Without NumPy | With NumPy |
|---------------|------------|
| Slow Python loops over millions of rows | C-optimized vectorized kernels |
| Lists hold mixed types | Homogeneous `dtype` (int64, float64) |
| Awkward linear algebra | `dot`, broadcasting, `axis` reductions |

Pandas Series and scikit-learn estimators ultimately rely on NumPy arrays. Today we use **restaurant vote counts** and **average meal costs** as toy features — the same patterns apply to Zomato columns in Labs 3–6.


---

## 1. Import NumPy and create arrays

The conventional alias is `import numpy as np`.


In [ ]:
import numpy as np

votes = np.array([120, 450, 890, 2100, 3400])
costs = np.array([400, 650, 1200, 1800, 2400])

print("votes:", votes)
print("costs:", costs)
print("votes dtype:", votes.dtype)
print("votes shape:", votes.shape)


### Shape and dtype

- **`shape`** — tuple of dimensions. `(5,)` means a 1-D array with 5 elements.
- **`dtype`** — `int64` here because we passed integers. Division later will promote to `float64`.

**Try it:** `len(votes)` equals `votes.shape[0]` for 1-D arrays.


---

## 2. Vectorized arithmetic

NumPy applies operators **element-wise** when shapes match.


In [ ]:
total_spend = votes * 0  # placeholder — use costs as spend proxy
revenue_proxy = costs * 1.1  # hypothetical 10% markup

print("Markup revenue (first 3):", np.round(revenue_proxy[:3], 2))

# Element-wise division — cost per vote (engagement efficiency)
cost_per_vote = costs / votes
print("cost_per_vote (first 3):", np.round(cost_per_vote[:3], 3))


Compare to a Python loop — vectorized code is shorter **and** faster at scale:

```python
# Slow pattern (avoid on large data)
result = []
for c, v in zip(costs, votes):
    result.append(c / v)
```

On **500** Zomato rows (Lab 3), vectorization matters.


---

## 3. Statistical normalization (z-score)

**Standardization** centers data at mean 0 and scale 1 — used before KNN (Day 4) and many ML models:

\[
z = rac{x - \mu}{\sigma}
\]

where \(\mu\) is the mean and \(\sigma\) is the sample standard deviation (`ddof=0` by default in NumPy).


In [ ]:
mean_votes = votes.mean()
std_votes = votes.std()

print(f"mean votes: {mean_votes}")
print(f"std votes:  {std_votes}")

normalized_votes = (votes - votes.mean()) / votes.std()
print("normalized_votes (first 3):", np.round(normalized_votes[:3], 3))
print("normalized mean (should ~0):", np.round(normalized_votes.mean(), 6))


**Interpretation:** The first restaurant (120 votes) is **below average** engagement → negative z-score (~ -1.05). The largest (3400 votes) is far above average.

This is the same idea as `StandardScaler` in scikit-learn (Day 3+).


---

## 4. Indexing and slicing

NumPy slicing returns a **view** (shared memory) for simple 1-D slices — be careful when mutating.


In [ ]:
print("votes[1:4]:", votes[1:4])
print("votes[-1]:", votes[-1])

# Boolean mask — restaurants with > 1000 votes
high_engagement = votes > 1000
print("mask:", high_engagement)
print("filtered costs:", costs[high_engagement])


---

## 5. Two-dimensional arrays with `column_stack`

Machine learning feature matrices are **2-D**: rows = samples, columns = features.


In [ ]:
matrix = np.column_stack([votes, costs])
print("matrix:\n", matrix)
print("matrix shape:", matrix.shape)
print("matrix ndim:", matrix.ndim)


### Axis reductions

| Expression | Meaning |
|------------|---------|
| `matrix.mean(axis=0)` | Mean **per column** (feature means) |
| `matrix.mean(axis=1)` | Mean **per row** (per restaurant) |
| `matrix.sum(axis=0)` | Sum per column |

`axis=0` collapses **rows** (vertical); `axis=1` collapses **columns** (horizontal).


In [ ]:
col_means = matrix.mean(axis=0)
row_means = matrix.mean(axis=1)

print("column means [votes, cost]:", np.round(col_means, 2))
print("row means (first 3):", np.round(row_means[:3], 2))
print("sum axis=0:", matrix.sum(axis=0))


---

## 6. Experiment — change one vote count

Edit `votes[2]` below and re-run normalization and `cost_per_vote`. Observe how **all** derived values update without rewriting loops.


In [ ]:
# Uncomment to experiment:
# votes_experiment = votes.copy()
# votes_experiment[2] = 5000
# print((votes_experiment - votes_experiment.mean()) / votes_experiment.std())

print("Current checkpoint uses original votes:", votes.tolist())


---

## 7. Final checkpoint


In [ ]:
print("Lab 2 — NumPy arrays")
print(f"votes shape: {votes.shape}, dtype: {votes.dtype}")
print(f"normalized_votes (first 3): {np.round(normalized_votes[:3], 3)}")
print(f"cost_per_vote (first 3): {np.round(cost_per_vote[:3], 3)}")
print(f"matrix shape: {matrix.shape}")
print(f"column means [votes, cost]: {np.round(col_means, 2)}")

assert votes.shape == (5,)
assert matrix.shape == (5, 2)
assert abs(col_means[0] - 1392) < 1 and abs(col_means[1] - 1290) < 1
print("\n✓ Checkpoint assertions passed")


---

## Reflection questions

1. Why is `votes.shape` written `(5,)` with a trailing comma concept but displayed that way?
2. When would `axis=1` be useful for a Zomato feature matrix?
3. How does `normalized_votes` relate to `StandardScaler` in scikit-learn?

**Previous:** [Lab 1 — Python structures](lab01_python_structures.ipynb)  
**Next:** [Lab 3 — Pandas Zomato load](lab03_pandas_zomato_load.ipynb)
